In [2]:
folder_path = "//Users/sahana/Documents/data_mining_db"

In [1]:
#Install required packages

import pandas as pd
import os
import unicodedata
import re
# !pip install langchain langchain-community langchain-groq sentence-transformers chromadb --break-system-packages
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import shutil

In [3]:
total_rows = 0

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder_path, file))
        rows = len(df)
        print(f"{file}: {rows} rows")
        total_rows += rows

print("Total rows in dataset:", total_rows)

Womens Innerwear.csv: 250 rows
Musical Instruments.csv: 250 rows
Mens Shirts.csv: 250 rows
Phones.csv: 170 rows
School Bags.csv: 250 rows
Refrigerators.csv: 250 rows
WomensFashion.csv: 250 rows
Mens Sports Shoes.csv: 250 rows
Womens Sandals.csv: 250 rows
Baby Products.csv: 250 rows
Toys and Games.csv: 250 rows
WesternWear.csv: 250 rows
Kids Clothing.csv: 250 rows
Motorbike Accessories.csv: 250 rows
Televisions.csv: 250 rows
Strength Training.csv: 250 rows
Mens Formal Shoes.csv: 250 rows
Handbags and Clutches.csv: 250 rows
Womens Watches.csv: 199 rows
Speakers.csv: 250 rows
Make-up.csv: 250 rows
Mens Innerwear.csv: 250 rows
Womens Shoes.csv: 250 rows
T-shirts and Polos.csv: 250 rows
Headphones.csv: 250 rows
Cameras.csv: 250 rows
Mens Watches.csv: 14 rows
Home Improvement.csv: 250 rows
Total rows in dataset: 6633


In [6]:
# Combining different CSV files from a data set into one single data frame.
df_list = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)

        df["source_file"] = file

        df_list.append(df)

# Combine all into one DataFrame
combined_df = pd.concat(df_list, ignore_index=True)

print("Total products after combining all product info from different csv files:", len(combined_df))
print()
print("Size of the data frame:", combined_df.shape)

Total products after combining all product info from different csv files: 6633

Size of the data frame: (6633, 7)


In [7]:
# Oitline of the dataset
combined_df.head()


,name,main_category,image,ratings,no_of_ratings,actual_price,source_file
0,Aston Andia Women Waist Cinchers Corset Adjust...,Womens Innerwear,https://m.media-amazon.com/images/I/51qLMr3Fea...,4.0,43.0,999.0,Womens Innerwear.csv
1,Floret Women's Cotton Non Padded Wire Free T-S...,Womens Innerwear,https://m.media-amazon.com/images/I/61sBzjDk7W...,4.0,1.0,399.0,Womens Innerwear.csv
2,DOT WAVE Women's Cotton Pyjamas - Loungepants ...,Womens Innerwear,https://m.media-amazon.com/images/I/81qQK0+n4p...,4.0,153.0,1699.0,Womens Innerwear.csv
3,Bali womens Comfort Revolution Wireless T-shir...,Womens Innerwear,https://m.media-amazon.com/images/I/81ReKuvZQr...,4.0,9328.0,1500.0,Womens Innerwear.csv
4,TRYLO Women's Non-Wired Bra,Womens Innerwear,https://m.media-amazon.com/images/W/IMAGERENDE...,4.0,11.0,435.0,Womens Innerwear.csv


In [24]:
# Checking for NaN values
combined_df.isnull().sum()


name             0
main_category    0
image            0
ratings          0
no_of_ratings    0
actual_price     0
source_file      0
dtype: int64

In [25]:
# Checking for empty or misiing values

print("Number of NaN values in each column:")
print(combined_df.isnull().sum())

print()
print("Number of missing values in each column:")
(combined_df == '').sum()

Number of NaN values in each column:
name             0
main_category    0
image            0
ratings          0
no_of_ratings    0
actual_price     0
source_file      0
dtype: int64

Number of missing values in each column:


name             0
main_category    0
image            0
ratings          0
no_of_ratings    0
actual_price     0
source_file      0
dtype: int64

In [8]:
# Checking for duplicate values and also dropping them

print("Number of Duplicates present: ",combined_df.duplicated().sum())

combined_df[combined_df.duplicated(keep=False)]

combined_df = combined_df.drop_duplicates(subset=['name', 'main_category', 'actual_price'])

print("Number of Duplicates present after clean up: ",combined_df.duplicated().sum())

Number of Duplicates present:  12
Number of Duplicates present after clean up:  0


In [8]:
print("Checking the data types of each column:")
print()
combined_df.info()

Checking the data types of each column:

<class 'pandas.DataFrame'>
RangeIndex: 6633 entries, 0 to 6632
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           6633 non-null   str    
 1   main_category  6633 non-null   str    
 2   image          6633 non-null   str    
 3   ratings        6633 non-null   float64
 4   no_of_ratings  6633 non-null   float64
 5   actual_price   6633 non-null   float64
 6   source_file    6633 non-null   str    
dtypes: float64(3), str(4)
memory usage: 362.9 KB


In [11]:
# Change the dtypr of the column no_of_ratings to int from float
combined_df['no_of_ratings'] = combined_df['no_of_ratings'].astype(int)

# Checking if all rating are valid
combined_df[(combined_df['ratings'] < 0) | (combined_df['ratings'] > 5)]
print("All the rating values are valid and are in the range 1 to 5.")
print()

# Change the dtypr of the column actual_price to int from float
combined_df['actual_price'] = combined_df['actual_price'].astype(int)

# Checking if the actual_price column has any -ve values
combined_df[combined_df['actual_price'] <= 0]
print("All the values in the column actual_price are valid, there is no 0 or -ve vales.")
print()

combined_df[["no_of_ratings", "actual_price"]].info()

USD_PER_INR = 0.011  # 1 INR ≈ 0.011 USD

combined_df['actual_price_usd'] = (combined_df['actual_price'] * USD_PER_INR).round(2)

All the rating values are valid and are in the range 1 to 5.

All the values in the column actual_price are valid, there is no 0 or -ve vales.

<class 'pandas.DataFrame'>
RangeIndex: 6633 entries, 0 to 6632
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   no_of_ratings  6633 non-null   int64
 1   actual_price   6633 non-null   int64
dtypes: int64(2)
memory usage: 103.8 KB


In [12]:
#  Standardizing Text Columns
combined_df['name'] = combined_df['name'].str.strip()

# Checking the categories in main_category
print("All values in column main_category:")
print()
print(combined_df['main_category'].unique())

# Changing the column name make-up to makeup
combined_df['main_category'] = combined_df['main_category'].replace({
    'make-up': 'makeup',
    't-shirts and polos': 'tshirts and polos',
})

All values in column main_category:

<StringArray>
[     'Womens Innerwear',   'Musical Instruments',           'Mens Shirts',
                'Phones',           'School Bags',         'Refrigerators',
        'Womens Fashion',     'Mens Sports Shoes',        'Womens Sandals',
         'Baby Products',                  'Toys',         'Kids Clothing',
 'Motorbike Accessories',           'Televisions',     'Strength Training',
     'Mens Formal Shoes', 'Handbags and Clutches',        'Womens Watches',
              'Speakers',               'Make-up',        'Mens Innerwear',
          'Womens Shoes',    'T-shirts and Polos',            'Headphones',
               'Cameras',          'Mens Watches',      'Home Improvement']
Length: 27, dtype: str


In [13]:
# search corpus

# Reset index and create product_id
combined_df = combined_df.reset_index(drop=True)
combined_df['product_id'] = combined_df.index

# Create searchable text corpus
combined_df['search_text'] = (
    'product: ' + combined_df['name'] + '. ' +
    'category: ' + combined_df['main_category'] + '. ' +
    'price: $' + combined_df['actual_price_usd'].astype(str) + '. ' +
    'rating: ' + combined_df['ratings'].astype(str) + ' out of 5. ' +
    'reviews: ' + combined_df['no_of_ratings'].astype(str) + '. ' +
    combined_df['name']
)

# Clean text for embeddings
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

combined_df['search_text'] = combined_df['search_text'].apply(clean_text)

print("Total number of columns: ",combined_df.columns.size)

print("Adter adding two new columns - product_id and search_text:")
print()

combined_df.head()

Total number of columns:  10
Adter adding two new columns - product_id and search_text:



,name,main_category,image,ratings,no_of_ratings,actual_price,source_file,actual_price_usd,product_id,search_text
0,Aston Andia Women Waist Cinchers Corset Adjust...,Womens Innerwear,https://m.media-amazon.com/images/I/51qLMr3Fea...,4.0,43,999,Womens Innerwear.csv,10.99,0,product: aston andia women waist cinchers cors...
1,Floret Women's Cotton Non Padded Wire Free T-S...,Womens Innerwear,https://m.media-amazon.com/images/I/61sBzjDk7W...,4.0,1,399,Womens Innerwear.csv,4.39,1,product: floret women's cotton non padded wire...
2,DOT WAVE Women's Cotton Pyjamas - Loungepants ...,Womens Innerwear,https://m.media-amazon.com/images/I/81qQK0+n4p...,4.0,153,1699,Womens Innerwear.csv,18.69,2,product: dot wave women's cotton pyjamas - lou...
3,Bali womens Comfort Revolution Wireless T-shir...,Womens Innerwear,https://m.media-amazon.com/images/I/81ReKuvZQr...,4.0,9328,1500,Womens Innerwear.csv,16.50,3,product: bali womens comfort revolution wirele...
4,TRYLO Women's Non-Wired Bra,Womens Innerwear,https://m.media-amazon.com/images/W/IMAGERENDE...,4.0,11,435,Womens Innerwear.csv,4.78,4,product: trylo women's non-wired bra. category...


In [14]:
# How sample serach_text looks for each row
print("Sample of how search_text lookds for each row/product: ")
print()
combined_df.iloc[0]["search_text"]

Sample of how search_text lookds for each row/product: 



'product: aston andia women waist cinchers corset adjustable hook&eye closure workout trainer flexible body shaper weight loss belt.... category: womens innerwear. price: $10.99. rating: 4.0 out of 5. reviews: 43. aston andia women waist cinchers corset adjustable hook&eye closure workout trainer flexible body shaper weight loss belt...'

In [16]:
# Cleaning text to create emdeding

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

combined_df['search_text'] = combined_df['search_text'].apply(clean_text)

combined_df.head()

,name,main_category,image,ratings,no_of_ratings,actual_price,source_file,actual_price_usd,product_id,search_text
0,Aston Andia Women Waist Cinchers Corset Adjust...,Womens Innerwear,https://m.media-amazon.com/images/I/51qLMr3Fea...,4.0,43,999,Womens Innerwear.csv,11.99,0,product: aston andia women waist cinchers cors...
1,Floret Women's Cotton Non Padded Wire Free T-S...,Womens Innerwear,https://m.media-amazon.com/images/I/61sBzjDk7W...,4.0,1,399,Womens Innerwear.csv,4.79,1,product: floret women's cotton non padded wire...
2,DOT WAVE Women's Cotton Pyjamas - Loungepants ...,Womens Innerwear,https://m.media-amazon.com/images/I/81qQK0+n4p...,4.0,153,1699,Womens Innerwear.csv,20.39,2,product: dot wave women's cotton pyjamas - lou...
3,Bali womens Comfort Revolution Wireless T-shir...,Womens Innerwear,https://m.media-amazon.com/images/I/81ReKuvZQr...,4.0,9328,1500,Womens Innerwear.csv,18.00,3,product: bali womens comfort revolution wirele...
4,TRYLO Women's Non-Wired Bra,Womens Innerwear,https://m.media-amazon.com/images/W/IMAGERENDE...,4.0,11,435,Womens Innerwear.csv,5.22,4,product: trylo women's non-wired bra. category...


In [17]:
#Remove special characters

combined_df['name'] = combined_df['name'].apply(lambda x: unicodedata.normalize('NFKD', str(x)))

In [18]:
# Checking the content lenght
combined_df['search_text'].str.len().describe()

count    6579.000000
mean      244.988448
std        69.484609
min        99.000000
25%       183.000000
50%       240.000000
75%       326.000000
max       346.000000
Name: search_text, dtype: float64

In [ ]:
# Savinf this cleaned dataset
combined_df.to_csv("clean_data.csv", index=False)